# Google ADK Browser Agent on AgentCore Runtime

## Overview

In this tutorial we will learn how to build a web research agent using **Google ADK** framework with **AgentCore Browser** and deploy it to **AWS Bedrock AgentCore Runtime**.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Deployment                                                                       |
| Agent type          | Single                                                                           |
| Agentic Framework   | Google ADK                                                                       |
| LLM model           | Anthropic Claude Haiku 4.5 (via Bedrock & LiteLLM)                               |
| Tutorial components | ADK Agent + AgentCore Browser + AgentCore Runtime deployment                     |
| Tutorial vertical   | Web Research                                                                     |
| Example complexity  | Medium                                                                           |
| SDK used            | Google ADK, Amazon BedrockAgentCore Python SDK, LiteLLM, Playwright              |

### Tutorial Architecture

In this tutorial we will describe how to deploy a Google ADK agent with browser capabilities to AgentCore Runtime.

```
┌─────────────────────────────────────────────────────────────┐
│                  AgentCore Runtime (Container)              │
│  ┌───────────────────────────────────────────────────────┐  │
│  │  Google ADK Agent                                     │  │
│  │  ├── LiteLLM → Bedrock Claude Haiku 4.5               │  │
│  │  └── Browser Tool → AgentCore Browser (Playwright)    │  │
│  └───────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

### Tutorial Key Features

* Using Google ADK with Bedrock models via LiteLLM
* Integrating AgentCore Browser for web browsing capabilities
* Live browser session viewing in AWS Console

## Prerequisites

### To execute this tutorial you will need:
* Python 3.10+
* AWS credentials configured (`aws configure`)
* AWS Account with Bedrock model access (Claude Haiku 4.5)
* IAM permissions for AgentCore Browser - see [Browser Permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config)

## 1. Install Dependencies

In [ ]:
!pip install -r requirements.txt --quiet

## 2. Create the Agent

Create the ADK agent with browser tool and AgentCore Runtime entrypoint:

In [ ]:
%%writefile adk_browser_agent.py
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from playwright.async_api import async_playwright
from bedrock_agentcore.tools.browser_client import browser_session
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import asyncio
import logging
import argparse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

APP_NAME = "adk_browser_agent"
REGION = "us-west-2"

async def browse_web(url: str) -> dict:
    """Browse websites using AgentCore Browser."""
    logger.info(f"browse_web called with url: {url}")
    try:
        with browser_session(REGION) as client:
            ws_url, headers = client.generate_ws_headers()
            async with async_playwright() as pw:
                browser = await pw.chromium.connect_over_cdp(ws_url, headers=headers)
                page = browser.contexts[0].pages[0]
                await page.goto(url, wait_until='domcontentloaded')
                result = {
                    'status': 'success',
                    'url': url,
                    'title': await page.title(),
                    'content': await page.inner_text('body')
                }
                await browser.close()
                return result
    except Exception as e:
        logger.error(f"browse_web error: {e}")
        return {'status': 'error', 'error': str(e)}

root_agent = LlmAgent(
    model=LiteLlm(model="bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0"),
    name="adk_browser_agent",
    instruction="You are a web research assistant. Use browse_web to visit URLs and extract information.",
    tools=[browse_web]
)

async def call_agent_async(prompt: str, user_id: str, session_id: str) -> str:
    """Run the ADK agent asynchronously."""
    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name=APP_NAME, user_id=user_id, session_id=session_id)
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=prompt)])
    
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.is_final_response():
            return event.content.parts[0].text
    return ""

# AgentCore Runtime entrypoint
app = BedrockAgentCoreApp()

@app.entrypoint
def agent_invocation(payload, context):
    logger.info(f"Invocation received: {payload}")
    result = asyncio.run(call_agent_async(
        payload.get("prompt", "Hello"),
        payload.get("user_id", "default"),
        context.session_id
    ))
    logger.info("Invocation complete")
    return result

if __name__ == "__main__":
    # Support both local testing and AgentCore Runtime
    import sys
    if len(sys.argv) > 1 and not sys.argv[1].startswith('--'):
        # Local testing mode
        prompt = sys.argv[1]
        result = asyncio.run(call_agent_async(prompt, "local-user", "local-session"))
        print(result)
    else:
        # AgentCore Runtime mode
        app.run()

## 3. Test Locally

Test the agent locally - it connects to the real AgentCore Browser:

In [ ]:
!python adk_browser_agent.py "Browse https://news.ycombinator.com and list the top 3 stories"

## 4. Configure AgentCore Deployment



In [ ]:
!agentcore configure \
    --name adk_browser_agent \
    --entrypoint adk_browser_agent.py \
    --region us-west-2 \
    --deployment-type container \
    --ecr auto \
    --disable-memory \
    --non-interactive

## 5. Deploy to AgentCore Runtime

This builds a container via CodeBuild and deploys to AgentCore (~5-10 min):

In [ ]:
!agentcore launch

In [ ]:
!agentcore status

## 6. Add Browser Permissions

The execution role needs browser permissions. Update the role permissions https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config

## 7. Invoke the Agent

Test the deployed agent:

In [ ]:
!agentcore invoke '{"prompt": "Hello, what can you do?"}'

In [ ]:
!agentcore invoke '{"prompt": "Browse https://news.ycombinator.com and list the top 3 stories"}'

## 8. View in AWS Console

### AgentCore Runtime Console
View your deployed agent, logs, and metrics:

👉 **[AgentCore Console](https://us-west-2.console.aws.amazon.com/bedrock-agentcore/home?region=us-west-2#/runtimes)**

### Live Browser View
Watch the browser in real-time while the agent browses:

👉 **[AgentCore Browser](https://us-west-2.console.aws.amazon.com/bedrock-agentcore/builtInTools?region=us-west-2)**

1. Go to **Built-in tools** → **Browser**
2. Find active session → Click **View live session**

## 9. Clean Up

Stop the session and destroy resources when done:

In [ ]:
!agentcore stop-session

In [ ]:
!agentcore destroy --force

## What happened behind the scenes?

* You created a Google ADK agent with a browser tool using AgentCore Browser
* The agent uses LiteLLM to connect to Bedrock Claude Haiku 4.5
* You tested locally with the AgentCore Browser tool
* You deployed to AgentCore Runtime using container deployment
* You invoked the deployed agent and watched it browse websites

## Summary

| Step | Command | Description |
|------|---------|-------------|
| Test Local | `python adk_browser_agent.py` | Test with real AgentCore Browser |
| Configure | `agentcore configure` | Set up container deployment |
| Deploy | `agentcore launch` | Build & deploy to AgentCore |
| Permissions | `aws iam put-role-policy` | Add browser permissions |
| Invoke | `agentcore invoke` | Invoke the deployed agent |
| Monitor | Console / `aws logs tail` | View logs and browser |
| Cleanup | `agentcore destroy` | Remove all resources |